# Benchmarking Harpia Module for Annotat3d Image Processing and Segmentation

This notebook benchmarks the **Harpia module**, the backend engine for Annotat3d, a tool designed for advanced image processing and segmentation tasks. The benchmarking process evaluates two key performance metrics:

1. **Accuracy**: Compares the results of Harpia’s image processing and segmentation functions with standard open-source tools like `scikit-image`, ensuring reliability and accuracy that meet industry standards.

2. **Execution Time**: Assesses runtime efficiency by measuring the execution times of Harpia’s functions relative to baseline tools, especially when handling large or complex images.

The benchmark is conducted on both **2D and 3D images** of various sizes, including very large images, to reflect Sirius demands in high-resolution and volumetric data efficient processing. This notebook provides a step-by-step analysis, including data setup, function execution, and results comparison, to evaluate Harpia's capability to handle a range of image dimensions and formats.


## Import Libraries

In [2]:
import numpy as np                     # For array manipulation
import pandas as pd                    # For data handling
import timeit                           # For timing the function
import matplotlib.pyplot as plt         # For plotting images

# Grayscale morphology operations
from skimage.morphology import (       
    erosion, dilation, closing, opening, 
    white_tophat, black_tophat
)

# Binary morphology operations
from skimage.morphology import binary_erosion, binary_dilation, binary_closing, binary_opening

# Workaround to allow importing harpia python module
import sys
sys.path.append("../../")

# Custom morphology operations from harpia for binary images
from harpia.morphology.operations_binary import (
     erosion_binary,
     dilation_binary,
     closing_binary,
     opening_binary,
     smooth_binary,
     geodesic_erosion_binary,
     geodesic_dilation_binary,
     reconstruction_binary,
     fill_holes
)

# Custom morphology operations from harpia for grayscale images
from harpia.morphology.operations_grayscale import (
     erosion_grayscale,
     dilation_grayscale,
     closing_grayscale,
     opening_grayscale,
     reconstruction_grayscale,
     top_hat,
     bottom_hat,
     top_hat_reconstruction,
     bottom_hat_reconstruction,
)

## Framework

In [3]:
def load_image(path, xsize, ysize, zsize, dtype, dtype_out):
    img = np.fromfile(path, dtype=dtype)
    img = img.reshape((zsize, ysize, xsize))
    img = img.astype(dtype = dtype_out)
    return img
    
def custum_kernel3D():
    kernel_2d = np.array([[1, 1, 1], [1, 1, 1], [1, 1, 1]], dtype=np.int32)
    # Stack the 2D kernel to form a 3D kernel (3 layers)
    kernel_3d = np.stack([kernel_2d, kernel_2d, kernel_2d])
    return kernel_3d

In [4]:
def time_module_only(csv_data, hardware, module_func, image, kernel, plot=False, operation="", slice_num=0, 
                     figsize=(18, 6), save_path=None, repetitions=1, *args, **kwargs):
    fontsize = 18
    times = []

    # Perform the function multiple times to average timing, ignoring the first run
    for _ in range(repetitions):
        start = timeit.default_timer()
        module_output = module_func(image, kernel, *args, **kwargs)
        times.append(timeit.default_timer() - start)

    # Calculate the mean time (ignoring the first warm-up run if repetitions > 1)
    if repetitions > 1:
        module_time = np.mean(times[1:])
    else:
        module_time = times[0]

    # Get the image data type, size, and dimensions
    image_dtype = str(image.dtype)
    image_size_bytes = image.nbytes
    image_shape = image.shape

    # Add timing information, data type, size, and dimensions to CSV data
    csv_data.append({
        'Operation': module_func.__name__ if not operation else operation,
        'Hardware': hardware,
        'Module Time (s)': module_time,
        'Scikit-Image Time (s)': 'N/A',
        'Accuracy': 'N/A',
        'Image Data Type': image_dtype,
        'Image Size (Bytes)': image_size_bytes,
        'Image Dimensions': image_shape
    })

    # Plot results if specified
    if plot:
        if len(image.shape) == 3:
            original_slice = image[slice_num, :, :]
            slice_module = module_output[slice_num, :, :]

        plt.figure(figsize=figsize)
        plt.subplot(1, 2, 1)
        plt.imshow(original_slice, cmap='gray')
        plt.title("Original Image", fontsize=fontsize)
        plt.axis('off')
        plt.subplot(1, 2, 2)
        plt.imshow(slice_module, cmap='gray')
        plt.title(f"Annotat3d {operation}", fontsize=fontsize)
        plt.axis('off')
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, bbox_inches='tight')
        plt.show()

    # Print timing information
    print(f"Operation: {module_func.__name__ if not operation else operation}")
    print(f"Module Time: {module_time:.4f} seconds")
    print(f"Image Data Type: {image_dtype}")
    print(f"Image Size: {image_size_bytes} bytes")
    print(f"Image Dimensions: {image_shape}")


# Merged function to time, compare, and plot results with MSE instead of accuracy
def time_compare_and_plot(csv_data, hardware, module_func, skimage_func, image, kernel, plot=False, 
                          operation="", framework="", slice_num=0, figsize=(18, 6), save_path=None, repetitions=1, *args, **kwargs):
    fontsize = 18
    module_times = []
    skimage_times = []

    # Time the module function multiple times
    for _ in range(repetitions):
        start = timeit.default_timer()
        module_output = module_func(image, kernel, *args, **kwargs)
        module_times.append(timeit.default_timer() - start)

    # Time the scikit-image function multiple times
    for _ in range(repetitions):
        start = timeit.default_timer()
        skimage_output = skimage_func(image, kernel, *args, **kwargs)
        skimage_times.append(timeit.default_timer() - start)

    # Calculate mean times, ignoring the first run if repetitions > 1
    if repetitions > 1:
        module_time = np.mean(module_times[1:])
        skimage_time = np.mean(skimage_times[1:])
    else:
        module_time = module_times[0]
        skimage_time = skimage_times[0]

    # Calculate Mean Squared Error
    mse = np.mean((skimage_output.astype(np.float32) - module_output.astype(np.float32)) ** 2)
    bitwise_diff = np.abs(skimage_output.astype(np.int32) - module_output.astype(np.int32))
    total_pixels = np.prod(image.shape)
    num_diff_pixels = np.count_nonzero(bitwise_diff)
    pixel_accuracy = ((total_pixels - num_diff_pixels) / total_pixels) * 100
    mean_diff = bitwise_diff.mean()
    std_diff = bitwise_diff.std()

    # Get the image data type, size, and dimensions
    image_dtype = str(image.dtype)
    image_size_bytes = image.nbytes
    image_shape = image.shape

    # Add timing, MSE, and image details to CSV data
    csv_data.append({
        'Operation': operation,
        'Gpus': hardware,
        'Module Time (s)': module_time,
        'Scikit-Image Time (s)': skimage_time,
        'Mean Squared Error': mse,
        'Pixel Accuracy (%)': pixel_accuracy,
        'Mean Difference': mean_diff,
        'Std Deviation of Difference': std_diff,
        'Image Data Type': image_dtype,
        'Image Size (Bytes)': image_size_bytes,
        'Image Dimensions': image_shape
    })

    # Plot results if specified
    if plot:
        if len(image.shape) == 3:
            original_slice = image[slice_num, :, :]
            slice_skimage = skimage_output[slice_num, :, :]
            slice_module = module_output[slice_num, :, :]

        plt.figure(figsize=figsize)
        plt.subplot(1, 3, 1)
        plt.imshow(original_slice, cmap='gray')
        plt.title("Original Image", fontsize=fontsize)
        plt.axis('off')
        plt.subplot(1, 3, 2)
        plt.imshow(slice_skimage, cmap='gray')
        plt.title(f"{framework} {operation}", fontsize=fontsize)
        plt.axis('off')
        plt.subplot(1, 3, 3)
        plt.imshow(slice_module, cmap='gray')
        plt.title(f"Annotat3d {operation}", fontsize=fontsize)
        plt.axis('off')
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, bbox_inches='tight')
        plt.show()

    # Print statistics
    print(f"Operation: {module_func.__name__ if not operation else operation}")
    print(f"Module Time: {module_time:.4f} seconds")
    print(f"Pixel Accuracy: {pixel_accuracy:.2f}%")
    print(f"Mean Squared Error: {mse:.2f}")
    print(f"Difference value: {mean_diff:.2f} ± {std_diff:.2f}")
    print(f"Image Data Type: {image_dtype}")
    print(f"Image Size: {image_size_bytes} bytes")
    print(f"Image Dimensions: {image_shape}")

In [5]:
def binarize_image(data, plot=False):
    zsize, ysize, xsize = data.shape  # Get dimensions
    binarized_data = np.empty_like(data, dtype = 'int32')  # Prepare output array of same shape

    for slice_idx in range(zsize):
        slice_data = data[slice_idx, :, :]

        # Find min and max for the current slice
        min_val = slice_data.min()
        max_val = slice_data.max()

        # Compute threshold
        threshold = (max_val + min_val) // 2

        # Apply threshold to the slice to create a binary image
        binarized_slice = np.where(slice_data >= threshold, 1, 0)

        # Store the binarized slice in the output array
        binarized_data[slice_idx, :, :] = binarized_slice

    # Plot the first slice if plot flag is True
    if plot:
        plt.figure(figsize=(10, 4))

        # Plot original first slice
        plt.subplot(1, 2, 1)
        plt.imshow(data[0, :, :], cmap='gray')
        plt.title('Original First Slice')
        plt.axis('off')

        # Plot binarized first slice
        plt.subplot(1, 2, 2)
        plt.imshow(binarized_data[0, :, :], cmap='gray')
        plt.title('Binarized First Slice')
        plt.axis('off')

        plt.show()

    return binarized_data

## Images

In [6]:
# original img
xsize = 2048
ysize = 2048
zsize = 1964
path = "../../../../../../../../beamlines/mogno/proposals/20180217/data/Soil_Experiment/testes_segmentacao/PBV29_Talita/tomoFiltered_masked_2048x2048x1964_16bit.raw"
image1_uint32 = load_image(path, xsize, ysize, zsize,'int16', 'uint32')
image1_int32 = load_image(path, xsize, ysize, zsize,'int16', 'int32')
image1_float32 = load_image(path, xsize, ysize, zsize,'int16', 'float32')

kernel = custum_kernel3D()

## Tests

In [9]:
csv_data = []
num_gpus = 1

In [7]:
help(time_compare_and_plot)

Help on function time_compare_and_plot in module __main__:

time_compare_and_plot(csv_data, hardware, module_func, skimage_func, image, kernel, plot=False, operation='', framework='', slice_num=0, figsize=(18, 6), save_path=None, repetitions=1, *args, **kwargs)
    # Merged function to time, compare, and plot results with MSE instead of accuracy



In [ ]:
time_compare_and_plot(csv_data, num_gpus, erosion_grayscale, erosion, image1_int32, kernel, plot=True, operation = "Erosion 3D", framework="scikit", 
    slice_num=1, save_path="plots/binary_erosion_"+str(image1_int32.dtype)+".png")

In [ ]:
for img in [image1_uint32, image1_int32, image1_float32]:
    time_compare_and_plot(csv_data, num_gpus, erosion_grayscale, erosion, img, kernel, plot=True, operation = "Erosion 3D", framework="scikit", 
    slice_num=1, save_path="plots/binary_erosion_"+str(img.dtype)+".png")

In [ ]:
'''
# Run and log each test
csv_data = []
num_gpus = 1

# Erosion Binary Test
time_and_compare(csv_data, num_gpus, erosion_binary, binary_erosion, image, kernel)

# Dilation Binary Test
time_and_compare(csv_data, num_gpus, dilation_binary, binary_dilation, image, kernel)

# Closing Binary Test
time_and_compare(csv_data, num_gpus, closing_binary, binary_closing, image, kernel)

# Opening Binary Test
time_and_compare(csv_data, num_gpus, opening_binary, binary_opening, image, kernela)

# Smooth binary, geodesic operations, and others without direct skimage equivalents
time_module_only(csv_data, num_gpus, smooth_binary, image, kernel)

# Add similar calls for other functions: geodesic_erosion_binary, geodesic_dilation_binary, reconstruction_binary, fill_holes

# Save results to CSV
results_df = pd.DataFrame(csv_data)
results_df.to_csv("morphology_benchmark_results.csv", index=False)

print("Benchmark completed and saved to morphology_benchmark_results.csv")
'''

In [ ]:
# Save results to CSV
results_df = pd.DataFrame(csv_data)
results_df.to_csv("morphology_benchmark_results.csv", index=False)

print("Benchmark completed and saved to morphology_benchmark_results.csv")

In [ ]:
results_df